# analysis_C — Pandas 평가/손익

- 담당: (이름)  브랜치: `<본인 github id>`
- 규칙: 커밋 전 `Kernel → Restart & Clear Output`. 이 노트북은 1인 전용.


In [ ]:
import pandas as pd
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
pf = pd.read_csv(ROOT/"data/portfolio.csv")
sp = pd.read_csv(ROOT/"data/stock_prices.csv")
pf.columns, sp.columns = pf.columns.str.strip().str.lower(), sp.columns.str.strip().str.lower()
sp["trade_date"] = pd.to_datetime(sp["trade_date"])
pf.head(), sp.head()

## 1. 데이터 이해 (컬럼, 결측, 중복)

In [ ]:
print(pf.info()); print(sp.info())
print(pf.isna().sum(), sp.isna().sum(), sep="\n")
print("dup:", pf.duplicated().sum(), sp.duplicated().sum())

## 2. 종목별 최신 종가 → 평가금액 / 손익 / 손익률  (SQL [A-2]와 동일 계산)

In [ ]:
last = sp.sort_values("trade_date").groupby("ticker").tail(1)[["ticker","close_price"]]
val = pf.merge(last, on="ticker")
val["cost_amount"] = val["quantity"] * val["buy_price"]
val["eval_amount"] = val["quantity"] * val["close_price"]
val["pnl"] = val["eval_amount"] - val["cost_amount"]
val["pnl_pct"] = ((val["close_price"] - val["buy_price"]) / val["buy_price"] * 100).round(2)
val["weight_pct"] = (val["eval_amount"] / val["eval_amount"].sum() * 100).round(2)
val = val.sort_values("pnl_pct", ascending=False)
val

## 3. 상위/하위 3 종목

In [ ]:
top3, bottom3 = val.head(3), val.tail(3)
top3, bottom3

## 4. 결과 내보내기 (검증 담당이 읽는 파일)

In [ ]:
out = ROOT/"outputs"; out.mkdir(exist_ok=True)
val.to_csv(out/"pandas_C_valuation.csv", index=False)
pd.concat([top3.assign(rank="top"), bottom3.assign(rank="bottom")]).to_csv(out/"pandas_C_top_bottom.csv", index=False)